# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

> **Reference**: [Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.](https://sen.science/doi/10.71728/senscience.vq4a-28xa)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets, their fields, and IDs. Each entity is referenced by its `@id` as per FAIR and Croissant best practices.


In [ ]:
# List the available RecordSets and their Field `@id` information
record_sets = list(dataset.record_sets)

print(f"Number of record sets: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {record_set.id}")
    print(f"     Name: {record_set.name if hasattr(record_set, 'name') else 'N/A'}")
    print(f"     Description: {record_set.description if hasattr(record_set, 'description') else 'N/A'}\n")

    print("     Fields:")
    for field in record_set.fields:
        print(f"        Field @id: {field.id} | Name: {field.name} | DataType: {field.data_type}")
    print("\n---------------------------\n")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# For demonstration, we typically extract main data record sets.
# Replace below with actual @id values after evaluating the overview above.

all_record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set @id: {record_set_id}, shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

# Select the primary data table for further exploration (manually choose; here we take the largest)
main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[1]) if len(dataframes) else None
print(f"\nChosen primary record set @id: {main_record_set_id}")
if main_record_set_id:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply typical data processing steps, such as filtering records by value, normalizing a numeric field, and grouping. All references are by their `@id` fields as listed previously.

In [ ]:
import numpy as np
from pandas.api.types import is_numeric_dtype

# Find a numeric field id in the main record set
df = dataframes[main_record_set_id]
numeric_field = None
for col in df.columns:
    if is_numeric_dtype(df[col]):
        numeric_field = col
        break
if not numeric_field:
    raise ValueError('No numeric field found in the primary record set.')

print(f"Numeric field chosen (by @id): {numeric_field}")

# Filtering records (choose threshold as e.g. the 75th percentile)
threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows")

# Normalization (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by another available field (e.g., first non-numeric column)
group_field = None
for col in df.columns:
    if not is_numeric_dtype(df[col]) and df[col].nunique() < 20 and col != numeric_field:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())
else:
    print("No suitable categorical field for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field is available, boxplot
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We successfully loaded the FAIR^2 mass spectrometry dataset via its Croissant schema using `mlcroissant`.
- We reviewed available record sets, extracted tabular data, and performed exploratory data analysis and visualization using only field and record set `@id` references.
- This approach supports reproducibility and clear referencing for future FAIR dataset workflows.

For deeper analysis, consider exploring additional fields, record sets, or linking with other protein databases as referenced in the dataset.